# YOLOv8-seg — Comparación: Tres Estrategias de Segmentación Vertebral

**Objetivo:** Comparar tres estrategias para segmentación vertebral T1–L5 en radiografías AP con escoliosis.

| | Modelo A | Modelo B | Modelo B+ |
|---|---|---|---|
| **Nombre** | Dataset completo | Dataset filtrado | Filtrado + Post-processing |
| **Imágenes train** | 174 (parciales incluidas) | ~92 (T1-L1 mínimo) | ~92 (igual que B) |
| **Post-processing** | Ninguno | Ninguno | Ajuste posicional por centroide Y |
| **Hipótesis** | Más datos ayudan | Calidad > cantidad | Morfología + posición ≈ razonamiento clínico |

**Contribución para la tesis:**
- **A vs B:** La completitud del campo visual es el factor determinante para segmentación lumbar.
- **B vs B+:** El ajuste posicional replica el razonamiento del radiólogo (morfología + posición relativa)
  y cuantifica su aporte incremental sobre el modelo base.

## 0 — Instalación y Setup

In [ ]:
!pip install -q ultralytics

from ultralytics import YOLO
import torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU:     {torch.cuda.get_device_name(0)}')
print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
import os, shutil, random, warnings
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from pathlib import Path
from sklearn.model_selection import train_test_split
from collections import Counter
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

# ── RUTAS ────────────────────────────────────────────────────
DRIVE_ROOT   = Path('/content/drive/MyDrive')
DATASET_ROOT = DRIVE_ROOT / 'Scoliosis_Dataset'
CSV_PATH     = DATASET_ROOT / 'indice_dataset.csv'
WORK_DIR     = Path('/content/yolo_spine')
RUNS_DIR     = WORK_DIR / 'runs'

# Datasets
YOLO_DS      = WORK_DIR / 'dataset'          # Dataset A: completo
YOLO_DS_FILT = WORK_DIR / 'dataset_filtered' # Dataset B: filtrado

# ── COLUMNAS CSV ─────────────────────────────────────────────
COL_SPLIT = 'split'
COL_IMAGE = 'radiograph_path'
COL_MASK  = 'multiclass_id_png'

# ── CLASES ───────────────────────────────────────────────────
CLASS_NAMES = [
    'T1','T2','T3','T4','T5','T6','T7','T8','T9','T10','T11','T12',
    'L1','L2','L3','L4','L5'
]
NUM_CLASSES     = 17
MASK_ID_TO_YOLO = {i: i-1 for i in range(1, 18)}

# ── HIPERPARÁMETROS ──────────────────────────────────────────
IMG_SIZE   = 1024
BATCH_SIZE = 8
SEED       = 42

random.seed(SEED)
np.random.seed(SEED)
for ds in [YOLO_DS, YOLO_DS_FILT]:
    for split in ['train', 'val', 'test']:
        (ds / 'images' / split).mkdir(parents=True, exist_ok=True)
        (ds / 'labels' / split).mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

print('✔ Configuración lista')

---
## 1 — Preparación del Dataset
### 1.1 — Carga y Split

In [ ]:
df = pd.read_csv(CSV_PATH, sep=';')
print(f'Total imágenes: {len(df)}')
print(df[COL_SPLIT].value_counts().to_string())

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df[COL_SPLIT], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df[COL_SPLIT], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'\nTrain: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')

### 1.2 — Conversión PNG → YOLO

In [ ]:
def mask_png_to_yolo(mask_path, mask_id_to_yolo, min_area=80, epsilon_factor=0.002):
    mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
    if mask is None: return []
    if mask.ndim == 3: mask = mask[:, :, 0]
    h, w = mask.shape
    lines = []
    for mask_id, yolo_cls in mask_id_to_yolo.items():
        binary = (mask == mask_id).astype(np.uint8) * 255
        if binary.sum() == 0: continue
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        contours = [c for c in contours if cv2.contourArea(c) >= min_area]
        if not contours: continue
        contour = max(contours, key=cv2.contourArea)
        eps    = epsilon_factor * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, eps, True)
        if len(approx) < 3: continue
        pts = approx.reshape(-1, 2).astype(np.float32)
        pts[:, 0] = np.clip(pts[:, 0] / w, 0.0, 1.0)
        pts[:, 1] = np.clip(pts[:, 1] / h, 0.0, 1.0)
        coords = ' '.join(f'{x:.6f} {y:.6f}' for x, y in pts)
        lines.append(f'{yolo_cls} {coords}')
    return lines


def build_yolo_dataset(split_dfs, output_dir, mask_id_to_yolo):
    """Convierte y copia imágenes+labels al directorio YOLO indicado."""
    stats = {}
    for split_name, split_df in split_dfs.items():
        ok = skip = 0
        for _, row in split_df.iterrows():
            img_src  = str(DATASET_ROOT / row[COL_IMAGE])
            mask_src = str(DATASET_ROOT / row[COL_MASK])
            stem     = Path(img_src).stem
            img_dst  = output_dir / 'images' / split_name / f'{stem}.jpg'
            lbl_dst  = output_dir / 'labels' / split_name / f'{stem}.txt'
            yolo_lines = mask_png_to_yolo(mask_src, mask_id_to_yolo)
            if not yolo_lines: skip += 1; continue
            img = cv2.imread(img_src)
            if img is None: skip += 1; continue
            cv2.imwrite(str(img_dst), img, [cv2.IMWRITE_JPEG_QUALITY, 95])
            lbl_dst.write_text('\n'.join(yolo_lines))
            ok += 1
        stats[split_name] = {'ok': ok, 'skip': skip}
        print(f'  {split_name:6s}: {ok} ok, {skip} omitidas')
    return stats


print('=== Construyendo Dataset A (completo) ===')
build_yolo_dataset(
    {'train': train_df, 'val': val_df, 'test': test_df},
    YOLO_DS, MASK_ID_TO_YOLO
)

# data.yaml Dataset A
YAML_A = YOLO_DS / 'data.yaml'
YAML_A.write_text(f"""path: {YOLO_DS}
train: images/train
val:   images/val
test:  images/test
nc: {NUM_CLASSES}
names: {CLASS_NAMES}
""")
print('✔ Dataset A listo')

### 1.3 — Dataset B: Filtrado por completitud de columna

In [ ]:
def get_classes_in_label(lbl_path):
    lines = Path(lbl_path).read_text().strip().split('\n')
    return set(int(l.split()[0]) for l in lines if l.strip())


def filter_complete_stems(lbl_dir, min_classes):
    """
    Retorna los stems de imágenes que tienen al menos `min_classes`
    clases consecutivas desde T1 (clase 0).
    min_classes=13 → T1-L1 mínimo (clases 0-12)
    """
    stems = []
    for lbl_path in sorted(Path(lbl_dir).glob('*.txt')):
        classes = get_classes_in_label(lbl_path)
        if all(c in classes for c in range(min_classes)):
            stems.append(lbl_path.stem)
    return stems


# Filtrar: mínimo T1-L1 (clases 0-12, 13 clases)
MIN_CLASSES = 13
train_stems = filter_complete_stems(YOLO_DS / 'labels' / 'train', MIN_CLASSES)
val_stems   = filter_complete_stems(YOLO_DS / 'labels' / 'val',   MIN_CLASSES)
test_stems  = filter_complete_stems(YOLO_DS / 'labels' / 'test',  MIN_CLASSES)

print(f'Dataset B — imágenes con al menos T1-L1:')
print(f'  Train: {len(train_stems)}/{len(train_df)}')
print(f'  Val:   {len(val_stems)}/{len(val_df)}')
print(f'  Test:  {len(test_stems)}/{len(test_df)}')

# Desglose por completitud en train
print('\nDesglose train Dataset B:')
with_l5  = sum(1 for s in train_stems
               if 16 in get_classes_in_label(YOLO_DS / 'labels' / 'train' / f'{s}.txt'))
print(f'  Con L5 completa (T1-L5)  : {with_l5}')
print(f'  Sin L5 (T1-L1 a T1-L4)  : {len(train_stems) - with_l5}')

In [ ]:
# Copiar al Dataset B
print('=== Construyendo Dataset B (filtrado) ===')
for split_name, stems in [('train', train_stems),
                            ('val',   val_stems),
                            ('test',  test_stems)]:
    ok = 0
    for stem in stems:
        src_img = YOLO_DS / 'images' / split_name / f'{stem}.jpg'
        src_lbl = YOLO_DS / 'labels' / split_name / f'{stem}.txt'
        dst_img = YOLO_DS_FILT / 'images' / split_name / f'{stem}.jpg'
        dst_lbl = YOLO_DS_FILT / 'labels' / split_name / f'{stem}.txt'
        if src_img.exists(): shutil.copy(src_img, dst_img); ok += 1
        if src_lbl.exists(): shutil.copy(src_lbl, dst_lbl)
    print(f'  {split_name:6s}: {ok} copiadas')

YAML_B = YOLO_DS_FILT / 'data.yaml'
YAML_B.write_text(f"""path: {YOLO_DS_FILT}
train: images/train
val:   images/val
test:  images/test
nc: {NUM_CLASSES}
names: {CLASS_NAMES}
""")
print('✔ Dataset B listo')

### 1.4 — Visualización comparativa de los datasets
Muestra cuántas imágenes tiene cada clase en cada dataset.

In [ ]:
def count_class_coverage(lbl_dir):
    """Cuenta cuántas imágenes tienen cada clase."""
    counts = {c: 0 for c in range(NUM_CLASSES)}
    total  = 0
    for lbl_path in Path(lbl_dir).glob('*.txt'):
        classes = get_classes_in_label(lbl_path)
        for c in classes:
            if c < NUM_CLASSES: counts[c] += 1
        total += 1
    return counts, total


cov_a, n_a = count_class_coverage(YOLO_DS      / 'labels' / 'train')
cov_b, n_b = count_class_coverage(YOLO_DS_FILT / 'labels' / 'train')

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
x = np.arange(NUM_CLASSES)
colors_a = ['#e74c3c' if cov_a[c]/n_a < 0.5 else '#3498db' for c in range(NUM_CLASSES)]
colors_b = ['#e74c3c' if cov_b[c]/n_b < 0.5 else '#2ecc71' for c in range(NUM_CLASSES)]

axes[0].bar(x, [100*cov_a[c]/n_a for c in range(NUM_CLASSES)], color=colors_a, edgecolor='k', alpha=0.85)
axes[0].axhline(50, color='red', ls='--', lw=1.5, label='50% umbral')
axes[0].set_xticks(x); axes[0].set_xticklabels(CLASS_NAMES, rotation=45)
axes[0].set_ylabel('% imágenes con esta clase'); axes[0].set_ylim(0, 110)
axes[0].set_title(f'Dataset A — Completo ({n_a} imgs train)\nRojo = < 50% cobertura')
axes[0].legend()

axes[1].bar(x, [100*cov_b[c]/n_b for c in range(NUM_CLASSES)], color=colors_b, edgecolor='k', alpha=0.85)
axes[1].axhline(50, color='red', ls='--', lw=1.5, label='50% umbral')
axes[1].set_xticks(x); axes[1].set_xticklabels(CLASS_NAMES, rotation=45)
axes[1].set_ylabel('% imágenes con esta clase'); axes[1].set_ylim(0, 110)
axes[1].set_title(f'Dataset B — Filtrado ({n_b} imgs train)\nVerde = ≥ 50% cobertura')
axes[1].legend()

plt.suptitle('Cobertura por clase en Train — Comparación datasets', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\nClases con < 50% cobertura:')
print(f'  Dataset A: {[CLASS_NAMES[c] for c in range(NUM_CLASSES) if cov_a[c]/n_a < 0.5]}')
print(f'  Dataset B: {[CLASS_NAMES[c] for c in range(NUM_CLASSES) if cov_b[c]/n_b < 0.5]}')

---
## 2 — Entrenamiento
### 2.1 — Modelo A: Dataset Completo (174 imágenes)

In [ ]:
print('=' * 60)
print('MODELO A — Dataset completo (174 imágenes train)')
print('=' * 60)

model_a = YOLO('yolov8m-seg.pt')
results_a = model_a.train(
    data          = str(YAML_A),
    epochs        = 150,
    patience      = 20,
    imgsz         = IMG_SIZE,
    batch         = BATCH_SIZE,
    device        = 0,
    seed          = SEED,
    project       = str(RUNS_DIR),
    name          = 'modelo_A_completo',
    exist_ok      = True,
    optimizer     = 'AdamW',
    lr0           = 0.01,
    lrf           = 0.001,
    warmup_epochs = 5,
    weight_decay  = 0.0005,
    degrees       = 15.0,
    translate     = 0.1,
    scale         = 0.3,
    shear         = 5.0,
    flipud        = 0.0,
    fliplr        = 0.5,
    mosaic        = 0.5,
    mixup         = 0.1,
    copy_paste    = 0.15,
    hsv_h         = 0.0,
    hsv_s         = 0.0,
    hsv_v         = 0.3,
    close_mosaic  = 30,
    workers       = 4,
    amp           = True,
    plots         = True,
    save          = True,
    save_period   = 10,
)

BEST_A = Path(results_a.save_dir) / 'weights' / 'best.pt'
print(f'\n✔ Modelo A completado → {BEST_A}')

### 2.2 — Modelo B: Dataset Filtrado (~92 imágenes)

In [ ]:
print('=' * 60)
print('MODELO B — Dataset filtrado (columnas con T1-L1 mínimo)')
print('=' * 60)

model_b = YOLO('yolov8m-seg.pt')
results_b = model_b.train(
    data          = str(YAML_B),
    epochs        = 200,          # más épocas — dataset más pequeño
    patience      = 30,
    imgsz         = IMG_SIZE,
    batch         = BATCH_SIZE,
    device        = 0,
    seed          = SEED,
    project       = str(RUNS_DIR),
    name          = 'modelo_B_filtrado',
    exist_ok      = True,
    optimizer     = 'AdamW',
    lr0           = 0.005,        # LR más bajo — dataset pequeño
    lrf           = 0.0005,
    warmup_epochs = 8,
    weight_decay  = 0.001,
    degrees       = 20.0,         # más augmentation para compensar
    translate     = 0.15,
    scale         = 0.4,
    shear         = 8.0,
    flipud        = 0.0,
    fliplr        = 0.5,
    mosaic        = 0.8,          # más mosaico
    mixup         = 0.15,
    copy_paste    = 0.3,          # copy_paste alto → sobremuestrea lumbares
    hsv_h         = 0.0,
    hsv_s         = 0.0,
    hsv_v         = 0.4,
    close_mosaic  = 40,
    workers       = 4,
    amp           = True,
    plots         = True,
    save          = True,
    save_period   = 10,
)

BEST_B = Path(results_b.save_dir) / 'weights' / 'best.pt'
print(f'\n✔ Modelo B completado → {BEST_B}')

---
## 3 — Evaluación
### 3.1 — Funciones de métricas

In [ ]:
def poly_to_mask(xy_norm, h, w):
    mask = np.zeros((h, w), dtype=np.uint8)
    pts  = xy_norm.copy()
    pts[:, 0] = np.clip(pts[:, 0] * w, 0, w-1)
    pts[:, 1] = np.clip(pts[:, 1] * h, 0, h-1)
    cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    return mask


def load_gt_from_txt(lbl_path, h, w, n_cls):
    gt = {c: np.zeros((h, w), dtype=np.uint8) for c in range(n_cls)}
    if not Path(lbl_path).exists(): return gt
    for line in Path(lbl_path).read_text().strip().split('\n'):
        parts = line.strip().split()
        if len(parts) < 7: continue
        c   = int(parts[0])
        pts = np.array(list(map(float, parts[1:]))).reshape(-1, 2)
        gt[c] = cv2.bitwise_or(gt[c], poly_to_mask(pts, h, w))
    return gt


def dice_iou(pred, gt):
    p, g  = pred.astype(bool), gt.astype(bool)
    inter = (p & g).sum()
    union = (p | g).sum()
    dice  = 2*inter/(p.sum()+g.sum()) if (p.sum()+g.sum()) > 0 else 1.0
    iou   = inter/union               if union > 0              else 1.0
    return float(dice), float(iou)


def evaluate_model(model, img_dir, lbl_dir, conf=0.25, model_name=''):
    """
    Evalúa Dice e IoU por clase sobre el test set.
    Usa siempre el GT del Dataset A (referencia canónica).
    """
    img_paths  = sorted(Path(img_dir).glob('*.jpg'))
    dice_cls   = {c: [] for c in range(NUM_CLASSES)}
    iou_cls    = {c: [] for c in range(NUM_CLASSES)}
    l5_details = []

    for img_path in img_paths:
        # GT siempre desde Dataset A (referencia completa)
        lbl_path = Path(lbl_dir) / f'{img_path.stem}.txt'
        result   = model.predict(
            str(img_path), imgsz=IMG_SIZE,
            conf=conf, device=0, verbose=False
        )[0]
        H, W = result.orig_shape
        gt   = load_gt_from_txt(str(lbl_path), H, W, NUM_CLASSES)

        pred = {c: np.zeros((H, W), dtype=np.uint8) for c in range(NUM_CLASSES)}
        if result.masks is not None:
            for i, cls_t in enumerate(result.boxes.cls):
                c   = int(cls_t.item())
                xy  = result.masks.xy[i]
                if len(xy) < 3: continue
                xyn = xy.copy()
                xyn[:, 0] /= W; xyn[:, 1] /= H
                pred[c] = cv2.bitwise_or(pred[c], poly_to_mask(xyn, H, W))

        for c in range(NUM_CLASSES):
            if gt[c].sum() == 0: continue
            d, iou = dice_iou(pred[c], gt[c])
            dice_cls[c].append(d)
            iou_cls[c].append(iou)
            if c == 16:
                l5_details.append({
                    'image'   : img_path.name,
                    'model'   : model_name,
                    'split'   : 'Scoliosis' if 'S_' in img_path.stem else 'Normal',
                    'dice'    : d, 'iou': iou,
                    'gt_px'   : int(gt[c].sum()),
                    'pred_px' : int(pred[c].sum()),
                    'detected': pred[c].sum() > 0
                })

    return dice_cls, iou_cls, l5_details


print('✔ Funciones de evaluación listas')

### 3.2 — Evaluar ambos modelos

In [ ]:
# Cargar mejores checkpoints
model_a_eval = YOLO(str(BEST_A))
model_b_eval = YOLO(str(BEST_B))

# GT de referencia: siempre Dataset A test (más imágenes)
TEST_IMG_DIR = YOLO_DS / 'images' / 'test'
TEST_LBL_DIR = YOLO_DS / 'labels' / 'test'

print('Evaluando Modelo A (completo)...')
dice_a, iou_a, l5_a = evaluate_model(
    model_a_eval, TEST_IMG_DIR, TEST_LBL_DIR,
    conf=0.25, model_name='A_completo'
)

print('Evaluando Modelo B (filtrado)...')
dice_b, iou_b, l5_b = evaluate_model(
    model_b_eval, TEST_IMG_DIR, TEST_LBL_DIR,
    conf=0.25, model_name='B_filtrado'
)

print('✔ Evaluación completada')

### 3.3 — Tabla comparativa por vértebra

In [ ]:
print(f'\n{"Clase":<6} {"Dice A":>8} {"Dice B":>8} {"Δ(B-A)":>8} {"IoU A":>8} {"IoU B":>8}')
print('─' * 52)

rows = []
for c in range(NUM_CLASSES):
    da = np.mean(dice_a[c]) if dice_a[c] else 0.0
    db = np.mean(dice_b[c]) if dice_b[c] else 0.0
    ia = np.mean(iou_a[c])  if iou_a[c]  else 0.0
    ib = np.mean(iou_b[c])  if iou_b[c]  else 0.0
    delta = db - da
    tag   = ' ◄ L5' if c == 16 else ''
    sign  = '+' if delta >= 0 else ''
    print(f'{CLASS_NAMES[c]:<6} {da:>8.4f} {db:>8.4f} {sign}{delta:>7.4f} {ia:>8.4f} {ib:>8.4f}{tag}')
    rows.append({'clase': CLASS_NAMES[c], 'dice_A': da, 'dice_B': db,
                 'delta': delta, 'iou_A': ia, 'iou_B': ib})

print('─' * 52)
mean_da = np.mean([r['dice_A'] for r in rows])
mean_db = np.mean([r['dice_B'] for r in rows])
mean_ia = np.mean([r['iou_A']  for r in rows])
mean_ib = np.mean([r['iou_B']  for r in rows])
print(f'{"MEAN":<6} {mean_da:>8.4f} {mean_db:>8.4f} {mean_db-mean_da:>+8.4f} '
      f'{mean_ia:>8.4f} {mean_ib:>8.4f}')

print(f'\n  Modelo A (completo)  → mean Dice: {mean_da:.4f} | mean IoU: {mean_ia:.4f}')
print(f'  Modelo B (filtrado)  → mean Dice: {mean_db:.4f} | mean IoU: {mean_ib:.4f}')
print(f'  Referencia paper semestre pasado → Dice: 0.74')

# Guardar tabla
results_df = pd.DataFrame(rows)
results_df.to_csv(DRIVE_ROOT / 'models' / 'comparacion_modelos.csv', index=False)
print('\n✔ Tabla guardada en Drive')

### 3.4 — Gráficos comparativos

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

x      = np.arange(NUM_CLASSES)
width  = 0.35
labels = CLASS_NAMES

# ── Gráfico 1: Dice por clase ─────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
bars_a = ax1.bar(x - width/2, [r['dice_A'] for r in rows],
                  width, label='Modelo A (completo)', color='#3498db', alpha=0.85, edgecolor='k')
bars_b = ax1.bar(x + width/2, [r['dice_B'] for r in rows],
                  width, label='Modelo B (filtrado)', color='#2ecc71', alpha=0.85, edgecolor='k')
ax1.axhline(0.70, color='red',  ls='--', lw=1.5, label='Umbral clínico=0.70')
ax1.axhline(0.74, color='gray', ls=':',  lw=1.5, label='Paper anterior=0.74')
ax1.set_xticks(x); ax1.set_xticklabels(labels, rotation=45)
ax1.set_ylabel('Dice'); ax1.set_ylim(0, 1.05)
ax1.set_title('Dice por vértebra — Modelo A vs Modelo B', fontsize=12)
ax1.legend(fontsize=9); ax1.grid(axis='y', alpha=0.3)

# Marcar L5
ax1.annotate('L5', xy=(16, max(rows[16]['dice_A'], rows[16]['dice_B']) + 0.03),
             ha='center', fontsize=9, color='red', fontweight='bold')

# ── Gráfico 2: Delta Dice (B - A) ────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
deltas = [r['delta'] for r in rows]
colors = ['#2ecc71' if d >= 0 else '#e74c3c' for d in deltas]
ax2.bar(x, deltas, color=colors, edgecolor='k', alpha=0.85)
ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=45)
ax2.set_ylabel('Δ Dice (B − A)')
ax2.set_title('Ganancia del filtrado por clase\nVerde=B mejor, Rojo=A mejor', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

# ── Gráfico 3: Resumen global ─────────────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
# Dividir en grupos: torácicas (T1-T12) y lumbares (L1-L5)
groups = {
    'T1–T6\n(cervico-\ntorácica)': range(0, 6),
    'T7–T12\n(torácica\nbaja)':    range(6, 12),
    'L1–L5\n(lumbar)':             range(12, 17),
    'GLOBAL':                      range(0, 17),
}
group_names = list(groups.keys())
vals_a = [np.mean([rows[c]['dice_A'] for c in rng]) for rng in groups.values()]
vals_b = [np.mean([rows[c]['dice_B'] for c in rng]) for rng in groups.values()]

xg = np.arange(len(group_names))
ax3.bar(xg - 0.2, vals_a, 0.35, label='Modelo A', color='#3498db', alpha=0.85, edgecolor='k')
ax3.bar(xg + 0.2, vals_b, 0.35, label='Modelo B', color='#2ecc71', alpha=0.85, edgecolor='k')
ax3.axhline(0.70, color='red',  ls='--', lw=1.5, label='Umbral=0.70')
ax3.axhline(0.74, color='gray', ls=':',  lw=1.5, label='Paper=0.74')
for i, (va, vb) in enumerate(zip(vals_a, vals_b)):
    ax3.text(i-0.2, va+0.01, f'{va:.2f}', ha='center', fontsize=8)
    ax3.text(i+0.2, vb+0.01, f'{vb:.2f}', ha='center', fontsize=8)
ax3.set_xticks(xg); ax3.set_xticklabels(group_names, fontsize=9)
ax3.set_ylabel('Dice promedio'); ax3.set_ylim(0, 1.05)
ax3.set_title('Dice por región anatómica', fontsize=11)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

plt.suptitle('Comparación: Dataset Completo vs Dataset Filtrado — YOLOv8m-seg',
             fontsize=14, fontweight='bold')
plt.savefig(str(DRIVE_ROOT / 'models' / 'comparacion_visual.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✔ Gráfico guardado en Drive')

### 3.5 — Análisis especial L5

In [ ]:
l5_df = pd.DataFrame(l5_a + l5_b)

print('═══ ANÁLISIS L5 ═══════════════════════════════════════')
summary = l5_df.groupby('model').agg(
    n_imgs    = ('image', 'count'),
    detectada = ('detected', 'mean'),
    dice_mean = ('dice', 'mean'),
    dice_med  = ('dice', 'median'),
    dice_std  = ('dice', 'std'),
    dice_min  = ('dice', 'min'),
    dice_max  = ('dice', 'max'),
).round(4)
print(summary.to_string())

print('\nDice L5 por modelo y tipo de columna:')
print(l5_df.groupby(['model', 'split'])['dice'].agg(
    ['mean','median','std','count']
).round(4).to_string())

# Gráfico L5
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Histogramas L5
for model_name, color in [('A_completo','#3498db'), ('B_filtrado','#2ecc71')]:
    sub = l5_df[l5_df['model'] == model_name]['dice']
    axes[0].hist(sub, bins=12, alpha=0.6, color=color,
                 label=f'{model_name} (μ={sub.mean():.3f})', edgecolor='k')
axes[0].axvline(0.70, color='red', ls='--', lw=2, label='Umbral=0.70')
axes[0].axvline(0.52, color='gray', ls=':', lw=1.5, label='Paper anterior=0.52')
axes[0].set_xlabel('Dice L5'); axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución Dice L5'); axes[0].legend(fontsize=8)

# Scatter: tamaño vs Dice
for model_name, color, marker in [('A_completo','#3498db','o'), ('B_filtrado','#2ecc71','s')]:
    sub = l5_df[l5_df['model'] == model_name]
    axes[1].scatter(sub['gt_px'], sub['dice'], alpha=0.7,
                    color=color, marker=marker, label=model_name, s=50)
axes[1].axhline(0.70, color='red', ls='--', lw=1.5)
axes[1].set_xlabel('Píxeles GT L5'); axes[1].set_ylabel('Dice L5')
axes[1].set_title('Dice L5 vs Tamaño de vértebra'); axes[1].legend(fontsize=8)

# Boxplot por modelo y tipo
import itertools
models  = ['A_completo', 'B_filtrado']
tipos   = ['Normal', 'Scoliosis']
bp_data = [l5_df[(l5_df['model']==m) & (l5_df['split']==t)]['dice'].values
           for m, t in itertools.product(models, tipos)]
bp_labels = [f'{m}\n{t}' for m, t in itertools.product(models, tipos)]
bp = axes[2].boxplot(bp_data, labels=bp_labels, patch_artist=True)
bp_colors = ['#3498db','#3498db','#2ecc71','#2ecc71']
for patch, color in zip(bp['boxes'], bp_colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[2].axhline(0.70, color='red', ls='--', lw=1.5, label='Umbral=0.70')
axes[2].set_ylabel('Dice L5')
axes[2].set_title('Dice L5: Normal vs Escoliosis\npor modelo')
axes[2].legend(fontsize=8); axes[2].tick_params(axis='x', labelsize=7)

plt.suptitle('Análisis Detallado L5 — Comparación Modelos', fontsize=13)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'models' / 'analisis_l5_comparacion.png'),
            dpi=150, bbox_inches='tight')
plt.show()

l5_df.to_csv(DRIVE_ROOT / 'models' / 'l5_comparacion.csv', index=False)
print('✔ Análisis L5 guardado en Drive')

### 3.6 — Curvas de entrenamiento comparativas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for model_name, results_obj, color, style in [
    ('A - Completo', results_a, '#3498db', '-'),
    ('B - Filtrado', results_b, '#2ecc71', '--'),
]:
    csv_path = Path(results_obj.save_dir) / 'results.csv'
    if not csv_path.exists(): continue
    r = pd.read_csv(csv_path)
    r.columns = r.columns.str.strip()
    ep = range(1, len(r)+1)

    if 'train/seg_loss' in r.columns:
        axes[0].plot(ep, r['train/seg_loss'], color=color, ls=style,
                     lw=1.5, label=f'Train {model_name}')
    if 'val/seg_loss' in r.columns:
        axes[0].plot(ep, r['val/seg_loss'], color=color, ls=':',
                     lw=1.5, label=f'Val {model_name}')
    if 'metrics/mAP50(M)' in r.columns:
        axes[1].plot(ep, r['metrics/mAP50(M)'], color=color, ls=style,
                     lw=2, label=model_name)
    if 'metrics/mAP50-95(M)' in r.columns:
        axes[2].plot(ep, r['metrics/mAP50-95(M)'], color=color, ls=style,
                     lw=2, label=model_name)

for ax, title in zip(axes, ['Seg Loss', 'mAP50 (Mask)', 'mAP50-95 (Mask)']):
    ax.set_title(title); ax.legend(fontsize=8)
    ax.grid(alpha=0.3); ax.set_xlabel('Época')

plt.suptitle('Curvas de Entrenamiento — Modelo A vs Modelo B', fontsize=13)
plt.tight_layout()
plt.savefig(str(DRIVE_ROOT / 'models' / 'curvas_entrenamiento.png'),
            dpi=150, bbox_inches='tight')
plt.show()

### 3.7 — Visualización cualitativa: misma imagen, dos modelos

In [ ]:
def visualize_comparison(model_a, model_b, img_path, lbl_path,
                          conf=0.25, title=''):
    """Muestra: Original | GT | Modelo A | Modelo B para una imagen."""
    img_orig = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W     = img_orig.shape[:2]
    rng      = np.random.RandomState(0)
    pal      = rng.randint(60, 230, (NUM_CLASSES, 3), dtype=np.uint8)
    gt       = load_gt_from_txt(str(lbl_path), H, W, NUM_CLASSES)

    def pred_overlay(model):
        res = model.predict(str(img_path), imgsz=IMG_SIZE,
                            conf=conf, device=0, verbose=False)[0]
        ov  = img_orig.copy()
        if res.masks is not None:
            for i, ct in enumerate(res.boxes.cls):
                c = int(ct.item())
                s = res.masks.xy[i].astype(np.int32)
                if len(s) < 3: continue
                col = tuple(int(x) for x in pal[c])
                colored = np.zeros_like(img_orig)
                cv2.fillPoly(colored, [s], col)
                ov = cv2.addWeighted(ov, 0.65, colored, 0.35, 0)
                cv2.polylines(ov, [s], True, col, 2)
                M = cv2.moments(s.astype(np.float32))
                if M['m00'] > 0:
                    cx = int(M['m10']/M['m00']); cy = int(M['m01']/M['m00'])
                    cv2.putText(ov, CLASS_NAMES[c], (cx-12, cy+5),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
        return ov

    def gt_overlay():
        ov = img_orig.copy()
        for c in range(NUM_CLASSES):
            if gt[c].sum() == 0: continue
            col = tuple(int(x) for x in pal[c])
            colored = np.zeros_like(img_orig)
            colored[gt[c]==1] = col
            ov = cv2.addWeighted(ov, 0.65, colored, 0.35, 0)
            cnts, _ = cv2.findContours(gt[c], cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(ov, cnts, -1, col, 2)
        return ov

    fig, axes = plt.subplots(1, 4, figsize=(22, 9))
    axes[0].imshow(img_orig);       axes[0].set_title('Original')
    axes[1].imshow(gt_overlay());   axes[1].set_title('Ground Truth')
    axes[2].imshow(pred_overlay(model_a)); axes[2].set_title('Modelo A\n(Dataset completo)')
    axes[3].imshow(pred_overlay(model_b)); axes[3].set_title('Modelo B\n(Dataset filtrado)')
    for ax in axes: ax.axis('off')
    plt.suptitle(title or Path(img_path).name, fontsize=12)
    plt.tight_layout()
    plt.show()


# Mostrar 4 ejemplos: 2 Normal, 2 Scoliosis
test_imgs = sorted((YOLO_DS / 'images' / 'test').glob('*.jpg'))
normal_imgs    = [p for p in test_imgs if p.stem.startswith('N_')]
scoliosis_imgs = [p for p in test_imgs if p.stem.startswith('S_')]

samples = random.sample(normal_imgs, min(2, len(normal_imgs))) + \
          random.sample(scoliosis_imgs, min(2, len(scoliosis_imgs)))

for img_path in samples:
    lbl_path = YOLO_DS / 'labels' / 'test' / f'{img_path.stem}.txt'
    tipo = 'Normal' if img_path.stem.startswith('N_') else 'Scoliosis'
    visualize_comparison(
        model_a_eval, model_b_eval,
        img_path, lbl_path,
        title=f'{img_path.name} ({tipo})'
    )

---
## 4 — Modelo B+: Post-processing Posicional

El Modelo B+ **no requiere reentrenamiento**. Usa exactamente los mismos pesos del Modelo B,
pero añade un paso de post-processing que resuelve la ambigüedad de identidad entre vértebras
visualmente similares.

### Fundamento clínico
Un radiólogo identifica las vértebras usando dos señales:
1. **Morfología** — forma, tamaño, presencia de costillas (aprendida por el modelo)
2. **Posición relativa** — T1 siempre está arriba, L5 siempre abajo (post-processing)

### Algoritmo
```
1. Obtener todas las detecciones del Modelo B (con confianza ≥ umbral)
2. Para cada clase duplicada → conservar solo la de mayor score
3. Ordenar detecciones supervivientes por centroide Y (↑ arriba → ↓ abajo)
4. Reasignar identidades según el orden anatómico conocido de la columna
   usando como ancla la clase predicha con mayor confianza
```

### Diferencia clave respecto a '1 clase + posición'
El Modelo B+ **no ignora lo que aprendió el modelo**. La reasignación usa
la predicción de clase como ancla inicial y solo corrige los casos donde
hay saltos o duplicados inconsistentes con la anatomía.

In [ ]:
# ============================================================
# POST-PROCESSING POSICIONAL — Modelo B+
# ============================================================

def positional_postprocessing(result, num_classes=17, conf=0.25):
    """
    Post-processing posicional sobre las predicciones del Modelo B.

    Estrategia:
    ─────────────────────────────────────────────────────────
    Paso 1 — Filtrar por confianza y eliminar duplicados
             Para cada clase, conservar solo la detección
             con mayor score (la más confiable morfológicamente).

    Paso 2 — Ancla semántica
             Encontrar la detección con mayor confianza absoluta.
             Esa clase predicha se considera correcta y actúa
             como punto de referencia (ancla).

    Paso 3 — Propagación posicional
             Ordenar todas las detecciones por centroide Y.
             A partir de la posición de la ancla, asignar
             identidades consecutivas hacia arriba y hacia abajo.
             Esto respeta el orden anatómico T1→L5 sin ignorar
             lo que aprendió el modelo.

    Retorna: dict {clase_reasignada: mascara_binaria}
    """
    if result.boxes is None or result.masks is None:
        return {}

    H, W = result.orig_shape

    # ── Paso 1: filtrar y eliminar duplicados ──────────────────
    best = {}  # {clase_predicha: {score, cy, mask}}
    for i, cls_t in enumerate(result.boxes.cls):
        c     = int(cls_t.item())
        score = float(result.boxes.conf[i].item())
        if score < conf:
            continue
        # Centroide Y del bounding box
        box = result.boxes.xyxy[i].cpu().numpy()
        cy  = float((box[1] + box[3]) / 2)
        # Convertir polígono a máscara
        xy  = result.masks.xy[i]
        if len(xy) < 3:
            continue
        xyn = xy.copy()
        xyn[:, 0] /= W
        xyn[:, 1] /= H
        mask = poly_to_mask(xyn, H, W)
        # Conservar solo la de mayor score por clase
        if c not in best or score > best[c]['score']:
            best[c] = {'score': score, 'cy': cy,
                       'mask': mask, 'orig_cls': c}

    if not best:
        return {}

    # ── Paso 2: ancla semántica ────────────────────────────────
    # La detección con mayor confianza es la más fiable morfológicamente
    anchor_cls = max(best, key=lambda c: best[c]['score'])
    anchor_cy  = best[anchor_cls]['cy']

    # ── Paso 3: propagación posicional ────────────────────────
    # Ordenar TODAS las detecciones por Y (arriba → abajo)
    dets_sorted = sorted(best.values(), key=lambda d: d['cy'])

    # Encontrar la posición del ancla en la lista ordenada
    anchor_pos = next(
        i for i, d in enumerate(dets_sorted)
        if abs(d['cy'] - anchor_cy) < 1.0
    )

    # Asignar clases: ancla mantiene su clase predicha,
    # el resto se asigna consecutivamente desde la ancla
    assigned = {}
    for offset, det in enumerate(dets_sorted):
        new_cls = anchor_cls + (offset - anchor_pos)
        if 0 <= new_cls < num_classes:
            assigned[new_cls] = det['mask']

    return assigned


print('✔ Función de post-processing posicional definida')
print()
print('Lógica del ancla semántica:')
print('  Si el modelo predice T6 con 0.92 de confianza (la más alta),')
print('  se asume que T6 es correcta. Las detecciones encima se reasignan')
print('  como T5, T4, T3... y las de abajo como T7, T8, T9...')
print('  Esto preserva el aprendizaje morfológico del modelo.')

In [ ]:
# ============================================================
# EVALUACIÓN MODELO B+
# ============================================================

def evaluate_bplus(model, img_dir, lbl_dir, conf=0.25, model_name='B+'):
    """
    Evalúa el Modelo B con post-processing posicional.
    Usa el mismo GT que A y B para comparación directa.
    """
    img_paths  = sorted(Path(img_dir).glob('*.jpg'))
    dice_cls   = {c: [] for c in range(NUM_CLASSES)}
    iou_cls    = {c: [] for c in range(NUM_CLASSES)}
    l5_details = []
    anchor_stats = []  # para análisis del ancla semántica

    for img_path in img_paths:
        lbl_path = Path(lbl_dir) / f'{img_path.stem}.txt'
        result   = model.predict(
            str(img_path), imgsz=IMG_SIZE,
            conf=conf, device=0, verbose=False
        )[0]
        H, W = result.orig_shape
        gt   = load_gt_from_txt(str(lbl_path), H, W, NUM_CLASSES)

        # Aplicar post-processing posicional
        assigned = positional_postprocessing(result, NUM_CLASSES, conf)

        # Registrar clase ancla para análisis
        if result.boxes is not None and len(result.boxes) > 0:
            best_idx   = result.boxes.conf.argmax().item()
            anchor_cls = int(result.boxes.cls[best_idx].item())
            anchor_conf= float(result.boxes.conf[best_idx].item())
            anchor_stats.append({
                'image': img_path.name,
                'anchor_cls': CLASS_NAMES[anchor_cls],
                'anchor_conf': anchor_conf
            })

        # Construir máscaras predichas desde el resultado post-procesado
        pred = {c: np.zeros((H, W), dtype=np.uint8) for c in range(NUM_CLASSES)}
        for cls_assigned, mask in assigned.items():
            pred[cls_assigned] = cv2.bitwise_or(pred[cls_assigned], mask)

        for c in range(NUM_CLASSES):
            if gt[c].sum() == 0:
                continue
            d, iou = dice_iou(pred[c], gt[c])
            dice_cls[c].append(d)
            iou_cls[c].append(iou)
            if c == 16:  # L5
                l5_details.append({
                    'image'   : img_path.name,
                    'model'   : model_name,
                    'split'   : 'Scoliosis' if 'S_' in img_path.stem else 'Normal',
                    'dice'    : d, 'iou': iou,
                    'gt_px'   : int(gt[c].sum()),
                    'pred_px' : int(pred[c].sum()),
                    'detected': pred[c].sum() > 0
                })

    return dice_cls, iou_cls, l5_details, anchor_stats


print('Evaluando Modelo B+ (filtrado + post-processing posicional)...')
dice_bp, iou_bp, l5_bp, anchor_stats = evaluate_bplus(
    model_b_eval,
    TEST_IMG_DIR, TEST_LBL_DIR,
    conf=0.25, model_name='B+'
)
print('✔ Evaluación Modelo B+ completada')

In [ ]:
# ── Análisis de la clase ancla ─────────────────────────────
# ¿Qué clase suele ser la más confiable? Esto valida el enfoque.
anchor_df = pd.DataFrame(anchor_stats)
print('Clase ancla más frecuente (clase con mayor confianza por imagen):')
print(anchor_df['anchor_cls'].value_counts().head(10).to_string())
print(f'\nConfianza promedio del ancla: {anchor_df["anchor_conf"].mean():.4f}')
print(f'Confianza mínima del ancla:   {anchor_df["anchor_conf"].min():.4f}')
print()
print('Nota para la tesis: Si las clases ancla más frecuentes son torácicas')
print('medias (T5-T8), esto confirma que el modelo aprende mejor las vértebras')
print('con morfología más distintiva, y el post-processing propaga esa certeza.')

### 4.1 — Tabla comparativa: A vs B vs B+

In [ ]:
print(f'\n{"Clase":<6} {"Dice A":>8} {"Dice B":>8} {"Dice B+":>9} '
      f'{"Δ(B-A)":>8} {"Δ(B+-B)":>9}')
print('─' * 56)

rows_3 = []
for c in range(NUM_CLASSES):
    da  = np.mean(dice_a[c])  if dice_a[c]  else 0.0
    db  = np.mean(dice_b[c])  if dice_b[c]  else 0.0
    dbp = np.mean(dice_bp[c]) if dice_bp[c] else 0.0
    d1  = db  - da
    d2  = dbp - db
    tag = ' ◄ L5' if c == 16 else ''
    print(f'{CLASS_NAMES[c]:<6} {da:>8.4f} {db:>8.4f} {dbp:>9.4f} '
          f'{d1:>+8.4f} {d2:>+9.4f}{tag}')
    rows_3.append({'clase': CLASS_NAMES[c],
                   'dice_A': da, 'dice_B': db, 'dice_Bplus': dbp,
                   'delta_B_A': d1, 'delta_Bplus_B': d2,
                   'iou_A': np.mean(iou_a[c])  if iou_a[c]  else 0.0,
                   'iou_B': np.mean(iou_b[c])  if iou_b[c]  else 0.0,
                   'iou_Bplus': np.mean(iou_bp[c]) if iou_bp[c] else 0.0})

print('─' * 56)
mean_da  = np.mean([r['dice_A']    for r in rows_3])
mean_db  = np.mean([r['dice_B']    for r in rows_3])
mean_dbp = np.mean([r['dice_Bplus'] for r in rows_3])
print(f'{"MEAN":<6} {mean_da:>8.4f} {mean_db:>8.4f} {mean_dbp:>9.4f} '
      f'{mean_db-mean_da:>+8.4f} {mean_dbp-mean_db:>+9.4f}')

print(f'\n  Modelo A  (completo)         → mean Dice: {mean_da:.4f}')
print(f'  Modelo B  (filtrado)         → mean Dice: {mean_db:.4f}')
print(f'  Modelo B+ (filtrado + pos.)  → mean Dice: {mean_dbp:.4f}')
print(f'  Paper semestre pasado        → mean Dice: 0.7400')

# Guardar tabla completa
df_3 = pd.DataFrame(rows_3)
df_3.to_csv(DRIVE_ROOT / 'models' / 'comparacion_3modelos.csv', index=False)
print('\n✔ Tabla guardada en Drive')

### 4.2 — Gráfico comparativo tres modelos

In [ ]:
fig = plt.figure(figsize=(20, 16))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)
x     = np.arange(NUM_CLASSES)
w     = 0.25

# ── Gráfico 1: Dice por clase (tres modelos) ──────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.bar(x - w,   [r['dice_A']    for r in rows_3], w,
        label='A — Completo',         color='#3498db', alpha=0.85, edgecolor='k')
ax1.bar(x,       [r['dice_B']    for r in rows_3], w,
        label='B — Filtrado',         color='#2ecc71', alpha=0.85, edgecolor='k')
ax1.bar(x + w,   [r['dice_Bplus'] for r in rows_3], w,
        label='B+ — Filtrado+Pos.',   color='#e67e22', alpha=0.85, edgecolor='k')
ax1.axhline(0.70, color='red',  ls='--', lw=1.5, label='Umbral clínico=0.70')
ax1.axhline(0.74, color='gray', ls=':',  lw=1.5, label='Paper anterior=0.74')
ax1.set_xticks(x); ax1.set_xticklabels(CLASS_NAMES, rotation=45)
ax1.set_ylabel('Dice'); ax1.set_ylim(0, 1.08)
ax1.set_title('Dice por vértebra — Tres estrategias comparadas', fontsize=12)
ax1.legend(fontsize=9, ncol=3); ax1.grid(axis='y', alpha=0.3)
ax1.annotate('L5', xy=(16, max(rows_3[16]['dice_A'],
             rows_3[16]['dice_B'], rows_3[16]['dice_Bplus'])+0.04),
             ha='center', fontsize=9, color='red', fontweight='bold')

# ── Gráfico 2: Aporte del post-processing (B+ - B) ───────────
ax2 = fig.add_subplot(gs[1, 0])
deltas_pp = [r['delta_Bplus_B'] for r in rows_3]
colors_pp = ['#e67e22' if d >= 0 else '#e74c3c' for d in deltas_pp]
ax2.bar(x, deltas_pp, color=colors_pp, edgecolor='k', alpha=0.85)
ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x); ax2.set_xticklabels(CLASS_NAMES, rotation=45)
ax2.set_ylabel('Δ Dice (B+ − B)')
ax2.set_title('Aporte del post-processing posicional\nNaranja=mejora, Rojo=empeora', fontsize=11)
ax2.grid(axis='y', alpha=0.3)

# ── Gráfico 3: Resumen por región anatómica ───────────────────
ax3 = fig.add_subplot(gs[1, 1])
groups = {
    'T1–T6\n(cervico-\ntorácica)': range(0, 6),
    'T7–T12\n(torácica\nbaja)':    range(6, 12),
    'L1–L5\n(lumbar)':             range(12, 17),
    'GLOBAL':                      range(0, 17),
}
gnames = list(groups.keys())
v_a  = [np.mean([rows_3[c]['dice_A']    for c in rng]) for rng in groups.values()]
v_b  = [np.mean([rows_3[c]['dice_B']    for c in rng]) for rng in groups.values()]
v_bp = [np.mean([rows_3[c]['dice_Bplus'] for c in rng]) for rng in groups.values()]
xg   = np.arange(len(gnames))

ax3.bar(xg - 0.25, v_a,  0.22, label='A',  color='#3498db', alpha=0.85, edgecolor='k')
ax3.bar(xg,        v_b,  0.22, label='B',  color='#2ecc71', alpha=0.85, edgecolor='k')
ax3.bar(xg + 0.25, v_bp, 0.22, label='B+', color='#e67e22', alpha=0.85, edgecolor='k')
ax3.axhline(0.70, color='red',  ls='--', lw=1.5, label='Umbral=0.70')
ax3.axhline(0.74, color='gray', ls=':',  lw=1.5, label='Paper=0.74')
for i, (va, vb, vbp) in enumerate(zip(v_a, v_b, v_bp)):
    ax3.text(i-0.25, va+0.01,  f'{va:.2f}',  ha='center', fontsize=8)
    ax3.text(i,      vb+0.01,  f'{vb:.2f}',  ha='center', fontsize=8)
    ax3.text(i+0.25, vbp+0.01, f'{vbp:.2f}', ha='center', fontsize=8)
ax3.set_xticks(xg); ax3.set_xticklabels(gnames, fontsize=9)
ax3.set_ylabel('Dice promedio'); ax3.set_ylim(0, 1.08)
ax3.set_title('Dice por región anatómica\nA vs B vs B+', fontsize=11)
ax3.legend(fontsize=8); ax3.grid(axis='y', alpha=0.3)

plt.suptitle('Comparación Completa — A (completo) vs B (filtrado) vs B+ (filtrado+posicional)',
             fontsize=13, fontweight='bold')
plt.savefig(str(DRIVE_ROOT / 'models' / 'comparacion_3modelos.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('✔ Gráfico guardado en Drive')

### 4.3 — Visualización cualitativa: Original | GT | A | B | B+

In [ ]:
def visualize_3models(model_a, model_b, img_path, lbl_path, conf=0.25, title=''):
    """Muestra: Original | GT | Modelo A | Modelo B | Modelo B+"""
    img_orig = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W     = img_orig.shape[:2]
    rng      = np.random.RandomState(0)
    pal      = rng.randint(60, 230, (NUM_CLASSES, 3), dtype=np.uint8)
    gt       = load_gt_from_txt(str(lbl_path), H, W, NUM_CLASSES)

    def make_overlay(masks_dict):
        ov = img_orig.copy()
        for c, mask in masks_dict.items():
            if mask.sum() == 0: continue
            col = tuple(int(x) for x in pal[c])
            colored = np.zeros_like(img_orig)
            colored[mask == 1] = col
            ov = cv2.addWeighted(ov, 0.65, colored, 0.35, 0)
            cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(ov, cnts, -1, col, 2)
            M = cv2.moments(mask)
            if M['m00'] > 0:
                cx = int(M['m10']/M['m00']); cy = int(M['m01']/M['m00'])
                cv2.putText(ov, CLASS_NAMES[c], (cx-12, cy+5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255,255,255), 1)
        return ov

    def pred_masks_raw(model):
        res  = model.predict(str(img_path), imgsz=IMG_SIZE,
                             conf=conf, device=0, verbose=False)[0]
        pred = {c: np.zeros((H, W), dtype=np.uint8) for c in range(NUM_CLASSES)}
        if res.masks is not None:
            for i, ct in enumerate(res.boxes.cls):
                c = int(ct.item())
                xy = res.masks.xy[i]
                if len(xy) < 3: continue
                xyn = xy.copy()
                xyn[:, 0] /= W; xyn[:, 1] /= H
                pred[c] = cv2.bitwise_or(pred[c], poly_to_mask(xyn, H, W))
        return pred

    def pred_masks_postproc(model):
        res      = model.predict(str(img_path), imgsz=IMG_SIZE,
                                 conf=conf, device=0, verbose=False)[0]
        assigned = positional_postprocessing(res, NUM_CLASSES, conf)
        pred = {c: np.zeros((H, W), dtype=np.uint8) for c in range(NUM_CLASSES)}
        for c, mask in assigned.items():
            pred[c] = cv2.bitwise_or(pred[c], mask)
        return pred

    gt_masks  = {c: gt[c] for c in range(NUM_CLASSES) if gt[c].sum() > 0}
    pred_a    = pred_masks_raw(model_a)
    pred_b    = pred_masks_raw(model_b)
    pred_bplus = pred_masks_postproc(model_b)

    fig, axes = plt.subplots(1, 5, figsize=(26, 9))
    axes[0].imshow(img_orig);              axes[0].set_title('Original')
    axes[1].imshow(make_overlay(gt_masks)); axes[1].set_title('Ground Truth')
    axes[2].imshow(make_overlay(pred_a));  axes[2].set_title('Modelo A\n(Completo)')
    axes[3].imshow(make_overlay(pred_b));  axes[3].set_title('Modelo B\n(Filtrado)')
    axes[4].imshow(make_overlay(pred_bplus)); axes[4].set_title('Modelo B+\n(Filtrado+Posicional)')
    for ax in axes: ax.axis('off')
    plt.suptitle(title or Path(img_path).name, fontsize=11)
    plt.tight_layout()
    plt.show()


# 4 ejemplos: 2 Normal, 2 Scoliosis
for img_path in samples:
    lbl_path = YOLO_DS / 'labels' / 'test' / f'{img_path.stem}.txt'
    tipo = 'Normal' if img_path.stem.startswith('N_') else 'Scoliosis'
    visualize_3models(
        model_a_eval, model_b_eval,
        img_path, lbl_path,
        title=f'{img_path.name} ({tipo})'
    )

---
## 5 — Guardar modelos en Drive

In [ ]:
save_dir = DRIVE_ROOT / 'models'
os.makedirs(save_dir, exist_ok=True)

dst_a = save_dir / 'yolov8m_A_completo_best.pt'
dst_b = save_dir / 'yolov8m_B_filtrado_best.pt'
shutil.copy(BEST_A, dst_a)
shutil.copy(BEST_B, dst_b)
# B+ usa los mismos pesos que B (el post-processing es en inferencia)

print(f'✔ Modelo A guardado: {dst_a}  ({dst_a.stat().st_size/1e6:.1f} MB)')
print(f'✔ Modelo B guardado: {dst_b}  ({dst_b.stat().st_size/1e6:.1f} MB)')
print(f'  Modelo B+ usa los mismos pesos que B + post-processing en inferencia')
print(f'\nArchivos guardados en Drive/models/:')
for f in sorted(save_dir.glob('*')):
    print(f'  {f.name}  ({f.stat().st_size/1e3:.0f} KB)')

---
## 6 — Resumen para la tesis

Esta celda genera el texto de resumen de resultados listo para copiar en el documento.

In [ ]:
def region_dice(dice_dict, cls_range):
    vals = [np.mean(dice_dict[c]) for c in cls_range if dice_dict[c]]
    return np.mean(vals) if vals else 0.0

l5_a_dice  = np.mean(dice_a[16])  if dice_a[16]  else 0.0
l5_b_dice  = np.mean(dice_b[16])  if dice_b[16]  else 0.0
l5_bp_dice = np.mean(dice_bp[16]) if dice_bp[16] else 0.0

print('=' * 65)
print('RESUMEN DE RESULTADOS — Para sección de resultados de la tesis')
print('=' * 65)
print(f"""
Se entrenaron y evaluaron tres variantes del modelo YOLOv8m-seg:

Modelo A (Dataset Completo):
  - Imágenes de entrenamiento : 174 (incluye columnas parciales)
  - mean Dice global          : {mean_da:.4f}
  - Dice torácicas (T1-T12)   : {region_dice(dice_a, range(0,12)):.4f}
  - Dice lumbares  (L1-L5)    : {region_dice(dice_a, range(12,17)):.4f}
  - Dice L5                   : {l5_a_dice:.4f}

Modelo B (Dataset Filtrado):
  - Imágenes de entrenamiento : {len(train_stems)} (T1-L1 mínimo)
  - mean Dice global          : {mean_db:.4f}
  - Dice torácicas (T1-T12)   : {region_dice(dice_b, range(0,12)):.4f}
  - Dice lumbares  (L1-L5)    : {region_dice(dice_b, range(12,17)):.4f}
  - Dice L5                   : {l5_b_dice:.4f}

Modelo B+ (Dataset Filtrado + Post-processing Posicional):
  - Mismos pesos que Modelo B
  - Post-processing: ancla semántica + propagación posicional
  - mean Dice global          : {mean_dbp:.4f}
  - Dice torácicas (T1-T12)   : {region_dice(dice_bp, range(0,12)):.4f}
  - Dice lumbares  (L1-L5)    : {region_dice(dice_bp, range(12,17)):.4f}
  - Dice L5                   : {l5_bp_dice:.4f}
  - Aporte del post-processing: {mean_dbp - mean_db:+.4f} Dice global

Referencia (paper semestre anterior, YOLOv8m, 134 imgs):
  - mean Dice global          : 0.7400
  - Dice L5                   : 0.5200

Hallazgos principales:
  1. La completitud del campo visual (A vs B) explica la mayor
     parte de la diferencia en Dice lumbar ({region_dice(dice_b, range(12,17))-region_dice(dice_a, range(12,17)):+.4f} en L1-L5).
  2. El post-processing posicional (B vs B+) aporta una mejora
     adicional de {mean_dbp-mean_db:+.4f} Dice global al resolver la ambigüedad
     de identidad entre vértebras visualmente similares.
  3. El enfoque B+ replica el razonamiento clínico del radiólogo:
     morfología aprendida por el modelo + posición relativa como
     señal de desambiguación.
""")
print('=' * 65)